In [ ]:
with Scene(model_frame) as scene:
    for center in centers:
        try:
            spectrum, morph = init.from_gaussian_moments(obs, center, min_corr=0.99)
        except ValueError:
            spectrum = init.pixel_spectrum(obs, center)
            morph = init.compact_morphology()

        Source(center, spectrum, morph)

In [ ]:
# coordinates of the transient
ra = 215.39425925333
dec = 37.90971372
coord = SkyCoord(ra, dec, unit="deg")

# separate channel information into band and epoch: 0 and 1 element
# depends on how channels encodes multi-epoch information
band_selector = lambda channel: channel[0]
epoch_selector = lambda channel: channel[1]

with scarlet2.Scene(model_frame) as scene:
    # 1) Host galaxy that is static across epochs
    try:
        spectrum, morph = scarlet2.init.from_gaussian_moments(obs, coord, box_sizes=[15, 21])
    except IndexError:
        morph = scarlet2.init.compact_morphology()
    # the host is barely resolved and the data are noisy:
    # use a starlet morphology for extra stability (esp to noise)
    morph = scarlet2.StarletMorphology.from_image(morph)

    # Select the transient-free epochs to initialize amplitudes for the static source
    # These will be shared across all epochs
    spectrum = spectrum[0:2]
    bands = ["ZTF_g", "ZTF_r"]
    scarlet2.Source(
        coord, scarlet2.StaticArraySpectrum(spectrum, bands=bands, band_selector=band_selector), morph
    )

    # 2) Point source for the transient, placed initially at same center
    # Define the epochs where the transient is allowed to have a non-zero amplitude
    epochs = [2, 3]
    # As we already know that the transient is present, we can measure the flux at the center location
    # This will be a mixture of host and transient light, to be corrected by the fitting procedure
    # Initializing as zero also works
    spectrum = scarlet2.init.pixel_spectrum(obs, coord)
    scarlet2.PointSource(
        coord, scarlet2.TransientArraySpectrum(spectrum, epochs=epochs, epoch_selector=epoch_selector)
    )

print(scene.sources)

In [ ]:
from numpyro.distributions import constraints

pos_step = 1e-2
morph_step = lambda p: scarlet2.relative_step(p, factor=1e-3)
SED_step = lambda p: scarlet2.relative_step(p, factor=5e-2)

parameters = scene.make_parameters()
# Static host galaxy parameters
parameters += scarlet2.Parameter(
    scene.sources[0].spectrum.data, name=f"spectrum.{0}", constraint=constraints.positive, stepsize=SED_step
)
parameters += scarlet2.Parameter(
    scene.sources[0].morphology.coeffs,
    name=f"morph.{0}",
    stepsize=morph_step,
)

# Transient point source parameters:
# no positive constraint on spectrum because it can be zero
parameters += scarlet2.Parameter(scene.sources[1].spectrum.data, name=f"spectrum.{1}", stepsize=SED_step)
parameters += scarlet2.Parameter(
    scene.sources[1].center, name=f"center.{1}", constraint=constraints.positive, stepsize=pos_step
)

In [ ]:
# Fit the scene
stepnum = 1000
scene_ = scene.fit(obs, parameters, max_iter=stepnum, e_rel=1e-4, progress_bar=False)